<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/MobilNet%20doble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cargar Base

In [2]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
path = kagglehub.dataset_download("leonardocaravaggio/ge-images")

mkdir: cannot create directory ‘/root/.kaggle’: File exists


100%|██████████| 13.3G/13.3G [03:04<00:00, 77.2MB/s]

Extracting files...


In [3]:
import pandas as pd
ciudades=pd.read_csv("base.csv")

In [4]:
len(ciudades)

1095

In [5]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm

# Modelo MobileNetV3-Large preentrenado
full_model = models.mobilenet_v3_large(pretrained=True)
mobilenet_v3 = full_model.features.eval()

# Transformaciones
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ✅ Función optimizada para extraer features (con (1,1))
def extract_features_global(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0)

    with torch.no_grad():
        features = mobilenet_v3(image)
        avg_pool = F.adaptive_avg_pool2d(features, (1, 1))  # (1,1)
        max_pool = F.adaptive_max_pool2d(features, (1, 1))
        features = torch.cat([avg_pool, max_pool], dim=1)
        features = features.view(-1).cpu().numpy()
    return features  # 1920 elementos

# Rutas
base_path = "/root/.cache/kagglehub/datasets/leonardocaravaggio/ge-images/versions/2/imagenes"
save_path = "/content/features_por_ciudad_global"
os.makedirs(save_path, exist_ok=True)

# Procesamiento de ciudades
for i in tqdm(range(len(ciudades))):
    nombre_archivo = ciudades.City[i].replace("/", ".").replace(":", "_").replace("'", "!")
    felicidad = ciudades.P1ST[i]

    for escala in ["1K", "10K"]:
        ruta_img = os.path.join(base_path, f"{nombre_archivo} - {escala}.png")

        if os.path.exists(ruta_img):
            try:
                features = extract_features_global(ruta_img)
                df_feat = pd.DataFrame(features).T
                df_feat.columns = [f"f{i}" for i in range(len(features))]
                df_feat["City"] = nombre_archivo
                df_feat["Escala"] = escala
                df_feat["P1ST"] = felicidad
                df_feat.to_csv(os.path.join(save_path, f"{nombre_archivo}_{escala}.csv"), index=False)
            except Exception as e:
                print(f"Error al procesar {ruta_img}: {e}")
        else:
            print(f"No encontrada: {ruta_img}")


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth
100%|██████████| 21.1M/21.1M [00:00<00:00, 111MB/s] 
100%|██████████| 1095/1095 [08:54<00:00,  2.05it/s]


In [6]:
import glob
import pandas as pd
from tqdm import tqdm

# Ruta donde guardaste los CSVs
ruta_features = "/content/features_por_ciudad_global"

# Buscar todos los archivos CSV
archivos_csv = glob.glob(os.path.join(ruta_features, "*.csv"))

# Leer y concatenar
df_features = pd.concat([pd.read_csv(f) for f in tqdm(archivos_csv)], ignore_index=True)
print("DataFrame unido:", df_features.shape)

100%|██████████| 2190/2190 [01:31<00:00, 24.01it/s]


DataFrame unido: (2190, 1923)


In [410]:
# Dividir el dataframe por escala
df_1km = df_features[df_features["Escala"] == "1K"]
df_10km = df_features[df_features["Escala"] == "10K"]

# Variables predictoras
X_1km = df_1km[[col for col in df_1km.columns if col.startswith("f")]]
y_1km = df_1km["P1ST"]

X_10km = df_10km[[col for col in df_10km.columns if col.startswith("f")]]
y_10km = df_10km["P1ST"]

# Modelos
modelo_1km = LinearRegression().fit(X_1km, y_1km)
modelo_10km = LinearRegression().fit(X_10km, y_10km)

print("R² 1 km:", r2_score(y_1km, modelo_1km.predict(X_1km)))
print("R² 10 km:", r2_score(y_10km, modelo_10km.predict(X_10km)))

R² 1 km: 0.9884115120679696
R² 10 km: 0.9997298582018904


In [412]:
def obtener_importancia_y_pesos(modelo, X):
    importancia = np.abs(modelo.coef_)
    top_indices = np.argsort(importancia)[::-1]
    pesos = modelo.coef_[top_indices]
    return top_indices, pesos

# Para el modelo de 1 km
features_indices_1k, pesos_1k = obtener_importancia_y_pesos(modelo_1km, X_1km)

# Para el modelo de 10 km
features_indices_10k, pesos_10k = obtener_importancia_y_pesos(modelo_10km, X_10km)


In [571]:
def extract_index(image_path, features_indices, pesos, alpha):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0)

    with torch.no_grad():
        features = mobilenet_v3(image)
        avg_pool = F.adaptive_avg_pool2d(features, (1, 1))
        max_pool = F.adaptive_max_pool2d(features, (1, 1))
        features = torch.cat([avg_pool, max_pool], dim=1)
        features = features.view(-1).numpy()

    top_features = features[features_indices]
    indice_modelo = np.dot(top_features, pesos)
    indice_visual = top_features.var()

    return alpha * indice_modelo + (1 - alpha) * indice_visual

def compute_inequality(image_1km_path, image_10km_path):
    indice_1km = extract_index(image_1km_path, features_indices_1k, pesos_1k, alpha=0.57)
    indice_10km = extract_index(image_10km_path, features_indices_10k, pesos_10k, alpha=0.1)
    diferencia = abs(indice_1km - indice_10km)

    return {
        "Desigualdad_1km": indice_1km,
        "Desigualdad_10km": indice_10km,
        "Diferencia": diferencia
    }


In [531]:
ciudades['Desigualdad_10km']=np.nan
ciudades['Desigualdad_1km']=np.nan
ciudades['Diferencia']=np.nan

#MobilNet v3

In [532]:
for i in range(len(ciudades)):
    if pd.isna(ciudades.loc[i, "Diferencia"]):
        try:
            nombre_archivo = ciudades.City[i].replace("/", ".").replace(":", "_").replace("'", "!")
            ruta_completa = os.path.join(path, "imagenes", nombre_archivo)

            img_1k = ruta_completa + " - 1K.png"
            img_10k = ruta_completa + " - 10K.png"

            if not os.path.exists(img_1k):
                print(f"❌ No existe: {img_1k}")
                continue

            resultados = compute_inequality(img_1k, img_10k)
            ciudades.loc[i, "Desigualdad_1km"] = resultados["Desigualdad_1km"]
            ciudades.loc[i, "Desigualdad_10km"] = resultados["Desigualdad_10km"]
            ciudades.loc[i, "Diferencia"] = resultados["Diferencia"]

        except Exception as e:
            print(f"⚠️ Error en {ciudades.City[i]}: {e}")

In [572]:
compute_inequality('/content/Oceano - 1K.png', '/content/Oceano - 10K.png')

{'Desigualdad_1km': np.float64(-144.1156103536075),
 'Desigualdad_10km': np.float64(0.22140264579288668),
 'Diferencia': np.float64(144.3370129994004)}

In [573]:
compute_inequality('/content/Amazonas - 1K.png', '/content/Amazonas - 10K.png')

{'Desigualdad_1km': np.float64(-86.63999099311981),
 'Desigualdad_10km': np.float64(-247.56386273200448),
 'Diferencia': np.float64(160.92387173888466)}

In [574]:
compute_inequality('/content/Cochabamba - 1K.png', '/content/Cochabamba - 10K.png')

{'Desigualdad_1km': np.float64(262.02567579984236),
 'Desigualdad_10km': np.float64(-147.74847823299788),
 'Diferencia': np.float64(409.7741540328402)}

In [552]:
compute_inequality('/content/Vitacura - 1K.png', '/content/Vitacura - 10K.png')

{'Desigualdad_1km': np.float64(-327.01609988477304),
 'Desigualdad_10km': np.float64(397.73391894590856),
 'Diferencia': np.float64(724.7500188306816)}

In [553]:
compute_inequality('/content/Favela Rocinha - 1K.png', '/content/Favela Rocinha - 10K.png')

{'Desigualdad_1km': np.float64(-31.600651201515078),
 'Desigualdad_10km': np.float64(189.3211468071153),
 'Diferencia': np.float64(220.9217980086304)}

In [547]:
compute_inequality('/content/Retiro - 1K.png', '/content/Retiro - 10K.png')

{'Desigualdad_1km': np.float64(2160.95779983657),
 'Desigualdad_10km': np.float64(2486.6359300480262),
 'Diferencia': np.float64(325.6781302114564)}

# Bajar la base con el indicador de desigualdad

In [ ]:
from google.colab import files
name="base_mobilv3_norm.csv"
ciudades.to_csv(name)
files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [533]:
import statsmodels.api as sm
import numpy as np

# Definir variables
X = ciudades["Desigualdad_10km"].replace([np.inf, -np.inf], np.nan)
y = ciudades["P1ST"].replace([np.inf, -np.inf], np.nan)

# Filtrar filas con NaN en X o y
mask = X.notna() & y.notna()
X, y = X[mask], y[mask]

# Agregar constante para la ordenada al origen
X = sm.add_constant(X)

# Ajustar modelo
modelo = sm.OLS(y, X).fit()

# Resumen de la regresión
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                   P1ST   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 4.044e+06
Date:                Fri, 11 Apr 2025   Prob (F-statistic):               0.00
Time:                        21:04:23   Log-Likelihood:                 4010.4
No. Observations:                1095   AIC:                            -8017.
Df Residuals:                    1093   BIC:                            -8007.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const              297.1759      0.146  